In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy joblib

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [3]:
import pandas as pd

data_path = "/kaggle/input/datasets/overgame1234/merged-text-tags-grouped/merged_text_tags_grouped.csv"
df = pd.read_csv(data_path)

In [4]:
df = df[["text", "Tags"]].copy()
df = df.dropna(subset=["text", "Tags"])

df["text"] = df["text"].astype(str).str.strip()
df["Tags"] = df["Tags"].astype(str).str.strip()

df = df[(df["text"] != "") & (df["Tags"] != "")]
print(df.shape)
df.head()

(1879950, 2)


,text,Tags
0,# + items .append is not a function codeblock ...,javascript
1,# - how to parallel code that lock several obj...,c#
2,# . what do and # do in this code codeblock tr...,javascript
3,# .dialog is not a function error after using ...,javascript
4,# .dialog is not a function error i am trying ...,javascript


In [5]:
df["label_list"] = df["Tags"].apply(lambda x: x.split())
df[["text", "Tags", "label_list"]].head()  


,text,Tags,label_list
0,# + items .append is not a function codeblock ...,javascript,[javascript]
1,# - how to parallel code that lock several obj...,c#,[c#]
2,# . what do and # do in this code codeblock tr...,javascript,[javascript]
3,# .dialog is not a function error after using ...,javascript,[javascript]
4,# .dialog is not a function error i am trying ...,javascript,[javascript]


In [6]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["label_list"])

print("Number of labels:", len(mlb.classes_))
print("y shape:", y.shape)
print("First labels:", mlb.classes_[:20])

Number of labels: 50
y shape: (1879950, 50)
First labels: ['active-directory' 'algorithm' 'amazon-ec2' 'android' 'apache' 'api'
 'architecture' 'bash' 'c#' 'c++' 'centos' 'data-structures'
 'database-design' 'debugging' 'design-patterns' 'dns' 'ftp' 'git' 'http'
 'image-processing']


In [7]:
X = df["text"].tolist()

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", len(X_train), y_train.shape)
print("Val  :", len(X_val), y_val.shape)
print("Test :", len(X_test), y_test.shape)

Train: 1503960 (1503960, 50)
Val  : 187995 (187995, 50)
Test : 187995 (187995, 50)


In [9]:
from datasets import Dataset
import numpy as np
y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)
train_size = min(400000, len(X_train))
val_size = min(40000, len(X_val))
test_size = min(40000, len(X_test))

rng = np.random.RandomState(42)

train_idx = rng.choice(len(X_train), train_size, replace=False)
val_idx = rng.choice(len(X_val), val_size, replace=False)
test_idx = rng.choice(len(X_test), test_size, replace=False)

X_train_sub = X_train.iloc[train_idx].tolist() if hasattr(X_train, "iloc") else [X_train[i] for i in train_idx]
X_val_sub = X_val.iloc[val_idx].tolist() if hasattr(X_val, "iloc") else [X_val[i] for i in val_idx]
X_test_sub = X_test.iloc[test_idx].tolist() if hasattr(X_test, "iloc") else [X_test[i] for i in test_idx]

y_train_sub = y_train[train_idx].tolist()
y_val_sub = y_val[val_idx].tolist()
y_test_sub = y_test[test_idx].tolist()

train_dataset = Dataset.from_dict({
    "text": X_train_sub,
    "labels": y_train_sub
})

val_dataset = Dataset.from_dict({
    "text": X_val_sub,
    "labels": y_val_sub
})

test_dataset = Dataset.from_dict({
    "text": X_test_sub,
    "labels": y_test_sub
})

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 400000
Val: 40000
Test: 40000


In [10]:
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/400000 [00:00<?, ? examples/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

In [12]:
keep_cols = ["input_ids", "attention_mask", "labels"]

train_dataset = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in keep_cols]
)
val_dataset = val_dataset.remove_columns(
    [col for col in val_dataset.column_names if col not in keep_cols]
)
test_dataset = test_dataset.remove_columns(
    [col for col in test_dataset.column_names if col not in keep_cols]
)

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

num_labels = y_train.shape[1]

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/roberta-base_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_at_3",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [15]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    results = {}

    for k in [1, 2, 3, 4]:
        preds_k = top_k_binary_predictions(logits, k)

        results[f"precision_at_{k}"] = precision_score(labels, preds_k, average="micro", zero_division=0)
        results[f"recall_at_{k}"] = recall_score(labels, preds_k, average="micro", zero_division=0)
        results[f"f1_at_{k}"] = f1_score(labels, preds_k, average="micro", zero_division=0)

    return results

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [17]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4
1,0.088753,0.047956,0.858925,0.757864,0.805236,0.505637,0.892288,0.645491,0.351900,0.931486,0.510821,0.269256,0.950302,0.419619
2,0.045343,0.041940,0.876875,0.773702,0.822064,0.514713,0.908303,0.657076,0.357242,0.945626,0.518575,0.272581,0.962037,0.424801
3,0.039550,0.040465,0.882125,0.778334,0.826986,0.517437,0.913112,0.660555,0.358408,0.948714,0.520268,0.273469,0.965170,0.426184
4,0.035977,0.039787,0.884275,0.780231,0.829001,0.518437,0.914876,0.661832,0.359008,0.950302,0.521139,0.273819,0.966405,0.426729


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=25000, training_loss=0.05240563537597656, metrics={'train_runtime': 22395.5993, 'train_samples_per_second': 71.443, 'train_steps_per_second': 1.116, 'total_flos': 1.052897796096e+17, 'train_loss': 0.05240563537597656, 'epoch': 4.0})

In [18]:
val_results = trainer.evaluate(eval_dataset=val_dataset)
print("Validation Results:", val_results)

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation Results: {'eval_loss': 0.03978651016950607, 'eval_precision_at_1': 0.884275, 'eval_recall_at_1': 0.7802311730709842, 'eval_f1_at_1': 0.829001335927063, 'eval_precision_at_2': 0.5184375, 'eval_recall_at_2': 0.9148762518198261, 'eval_f1_at_2': 0.6618315860022022, 'eval_precision_at_3': 0.3590083333333333, 'eval_recall_at_3': 0.9503022014382141, 'eval_f1_at_3': 0.521139027665211, 'eval_precision_at_4': 0.27381875, 'eval_recall_at_4': 0.9664049058102087, 'eval_f1_at_4': 0.4267291339963182, 'eval_runtime': 178.6112, 'eval_samples_per_second': 223.95, 'eval_steps_per_second': 3.499, 'epoch': 4.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test Results: {'eval_loss': 0.04084701091051102, 'eval_precision_at_1': 0.8801, 'eval_recall_at_1': 0.7743780383185588, 'eval_f1_at_1': 0.8238611764430559, 'eval_precision_at_2': 0.51835, 'eval_recall_at_2': 0.9121664723609247, 'eval_f1_at_2': 0.6610500474250962, 'eval_precision_at_3': 0.36004166666666665, 'eval_recall_at_3': 0.9503750467433625, 'eval_f1_at_3': 0.5222378687424831, 'eval_precision_at_4': 0.2746, 'eval_recall_at_4': 0.9664547634235938, 'eval_f1_at_4': 0.42768213918943254, 'eval_runtime': 179.1661, 'eval_samples_per_second': 223.256, 'eval_steps_per_second': 3.488, 'epoch': 4.0}


In [ ]:
model.save_pretrained("/kaggle/working/deberta_v3_best")
tokenizer.save_pretrained("/kaggle/working/deberta_v3_best")
joblib.dump(mlb, "/kaggle/working/deberta_v3_best/mlb.pkl")

print("Model, tokenizer, and label binarizer saved successfully.")